# MMIRAG — follow-up answers and faithfulness audit (runs A and B; about US$0.55, hard limit US$0.95)

* **Run A.** On MedMCQA, 19 replies could not be read as an answer (7 No-RAG, 7 single-pass, 5 decoupled), almost all bare gap statements. Each gets the same one-time request for a final answer that RAG-SEQ already gets at the cap, so all conditions are scored the same way.
* **Run B.** A different model (gpt-4o-mini) judges whether each RAG-SEQ justification is supported by the passages RAG-SEQ showed the reader, on all 250 primary and 1,000 MedMCQA questions, with the same judging prompt and rule as the earlier audit.

**How to run:** put your OpenAI key in the Colab *Secrets* panel as `OPENAI_API_KEY` (key icon on the left, notebook access on), then *Runtime → Run all*. Everything before the "Runs A and B" cell is free: it reloads the saved knowledge-base index and the saved replies from `MMIRAG_R3` on Drive.

**Nothing is lost and nothing is paid twice.** Every reply and every result is written to `MMIRAG_R3` on Drive the moment it arrives; if the run stops (credit, budget, disconnect), run all again and it continues where it stopped. The original result files, including `api_cache.jsonl`, are not modified, so their fingerprints in `MANIFEST.md` stay valid. This notebook has no question-building section, so `clean250.csv` and `medmcqa_set.csv` cannot be rewritten.

Check in the output: *"original call matched: True"* on every line of run A, and *"context rebuilt exactly for 250"* (then 1000) in run B. The last cell copies a zip of the new files to Drive and downloads it.

## 1. Setup

In [1]:
!pip -q install openai sentence-transformers faiss-cpu pymupdf datasets openpyxl

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORKDIR = '/content/drive/MyDrive/MMIRAG_R3'     # results, knowledge base and caches live here
os.makedirs(f'{WORKDIR}/kb/pdfs', exist_ok=True)
REPO = '/content/MMIRAG'
if not os.path.exists(REPO):
    !git clone -q https://github.com/youssefAZAMI360/MMIRAG {REPO}
print(sorted(os.listdir(REPO)))

Mounted at /content/drive
['.git', 'README.md', 'data', 'evaluation', 'prompts', 'results', 'retrieval']


In [3]:
# OpenAI key: Colab "Secrets" panel (key icon, left) -> add OPENAI_API_KEY and allow notebook access.
from google.colab import userdata
from getpass import getpass
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')
print('key set')

OPENAI_API_KEY: ··········
key set


## 2. Settings and prompts (prompts are read from the repository)

In [4]:
# price = (input, output, cached input) in US$ per million tokens, OpenAI list prices
READERS = {
    'gpt-4.1-mini': dict(model='gpt-4.1-mini', price=(0.40, 1.60, 0.10)),   # main reader, as in the paper's replication
    # Optional weaker reader for the reader-strength comparison (adds about US$2; gpt-4.1-nano is retired):
    # 'gpt-4o-mini': dict(model='gpt-4o-mini', price=(0.15, 0.60, 0.075)),
}
MAIN_READER = 'gpt-4.1-mini'       # the only reader that also runs the decoupled-assessor condition
ASSESSOR_MODEL, ASSESSOR_PRICE = 'gpt-4o-mini', (0.15, 0.60, 0.075)   # a different model decides when to stop

N_PER_DOMAIN = 250                 # MedMCQA questions per domain (5 domains -> 1,250); total cost about US$7.5
CAP, TOP_K = 3, 5                  # repository README: max 3 iterations, top-5 per retrieval
EMBED_MODEL = 'sentence-transformers/all-mpnet-base-v2'    # repository README (768-d)
CHUNK_WORDS, CHUNK_OVERLAP = 180, 40                         # repository README
WORKERS = 8                        # lower to 4 if you see many rate-limit retries
MAX_USD = 9.00                     # hard spending cap for this notebook: no API call is made beyond it

In [5]:
import re
def _read(name):
    t = open(f'{REPO}/prompts/{name}', encoding='utf-8').read().replace('\r\n', '\n')
    return '\n'.join(l for l in t.split('\n') if not l.lstrip().startswith('#')).strip() + '\n'

SYSTEM_PROMPT = _read('system_prompt.txt').rstrip() + (
    "\n- After the justification, end your reply with a separate last line: FINAL ANSWER: <letter>"
    "\n- If you reply with GAP_IDENTIFIED instead of answering, do not give an answer or a FINAL ANSWER line.\n")
T_NORAG = _read('no_rag_prompt.txt'); T_SP = _read('rag_single_pass_prompt.txt')
T_IT1 = _read('rag_seq_iteration1.txt'); T_ITN = _read('rag_seq_iterationN.txt')

def _fill_q(t, q):
    letters = [L for L in 'ABCDE' if str(q.get(L, '')).strip()]
    opts = '\n'.join(f'{L}. {q[L]}' for L in letters)
    t = re.sub(r'A\. \{option_a\}.*?E\. \{option_e\}[^\n]*', lambda m: opts, t, flags=re.S)
    hist = str(q.get('history', '') or '').strip()
    t = t.replace('Patient History:\n{patient_history}\n\n', '' if not hist else f'Patient History:\n{hist}\n\n')
    return t.replace('{question_stem}', str(q['question']).strip()), letters

def _chunks(ps):
    return '\n\n'.join(p.strip() for p in ps)

def p_norag(q):
    return _fill_q(T_NORAG, q)

def p_single(q, passages):
    t, L = _fill_q(T_SP, q)
    t = re.sub(r'\{retrieved_chunk_1\}.*?\{retrieved_chunk_3\}', lambda m: _chunks(passages), t, flags=re.S)
    return t, L

def p_iter1(q, passages):
    t, L = p_single(q, passages)
    return t.rstrip() + '\n\n' + T_IT1, L

def p_iterN(q, gap, passages):
    t, L = _fill_q(T_ITN, q)
    t = t.replace('{gap_description_from_previous_iteration}', gap)
    t = re.sub(r'\{new_retrieved_chunk_1\}.*?\{new_retrieved_chunk_2\}', lambda m: _chunks(passages), t, flags=re.S)
    return t, L


ASSESSOR_PROMPT = """You are checking whether retrieved evidence is sufficient to answer a
pathology multiple-choice question. Do NOT answer the question.

--- RETRIEVED CONTEXT ---
{context}
--- END RETRIEVED CONTEXT ---

Question:
{question}

Options:
{options}

If the context contains the information needed to choose an option, reply exactly:
SUFFICIENT
Otherwise reply exactly:
GAP_IDENTIFIED: <what to search for next, in one sentence>"""

FORCE_MSG = ("You have reached the maximum number of retrieval rounds. Using the question, the options "
             "and all the context above, choose the single best option now.\n"
             "Format: [LETTER] [Justification], then a last line FINAL ANSWER: <letter>")

_q = dict(question='Example stem?', A='one', B='two', C='three', D='four', E='')
print(SYSTEM_PROMPT); print('-' * 60); print(p_iter1(_q, ['passage 1', 'passage 2'])[0])

You are an expert medical professional specialising in pathology.

Your task is to answer multiple-choice questions about medical pathology.

You must reason step by step through the clinical and pathological evidence
before selecting your answer.

Rules:
- Read the full question stem and all answer options carefully.
- If a pathological image is provided, integrate its visual findings
  into your reasoning.
- Return ONLY a single capital letter (A, B, C, D, or E) as your answer,
  followed by a concise medical justification of 2-4 sentences.
- Do not add any text before the answer letter.
- Format: [LETTER] [Justification]
- After the justification, end your reply with a separate last line: FINAL ANSWER: <letter>
- If you reply with GAP_IDENTIFIED instead of answering, do not give an answer or a FINAL ANSWER line.

------------------------------------------------------------
Answer the following pathology multiple-choice question.
Use the retrieved context below to support your reason

## 3. Knowledge base
Uses the same corpus as the earlier `gpt-4.1-mini` runs (retrieval ablation and cap sweep): every PDF in the Drive folder **`robbins_pdfs`** (it contains `CAP11.pdf`), chunked exactly as those notebooks did (180 words, 40-word overlap, PyMuPDF text), embedded with `all-mpnet-base-v2`, FAISS `IndexFlatL2`. Those runs produced 1,398 chunks; you should see the same number. The index is saved to `MMIRAG_R3/kb/`, so this runs once.

Do **not** add *Robbins and Cotran Review of Pathology* (a question book) to this folder: retrieving from a book of practice questions could leak answers.

In [6]:
import json, glob, re, threading, numpy as np, pandas as pd, faiss, fitz
from sentence_transformers import SentenceTransformer
# The corpus of the earlier gpt-4.1-mini runs (retrieval ablation, cap sweep):
# their notebooks read every PDF in this Drive folder (it contains CAP11.pdf) and produced 1,398 chunks.
KB_FOLDER = '/content/drive/MyDrive/robbins_pdfs'
INDEX_PATH, CHUNKS_PATH = f'{WORKDIR}/kb/index.faiss', f'{WORKDIR}/kb/chunks.json'
os.makedirs(f'{WORKDIR}/kb', exist_ok=True)
embedder = SentenceTransformer(EMBED_MODEL)

def chunk(t, w=CHUNK_WORDS, o=CHUNK_OVERLAP):      # identical to the chunker of the earlier notebooks
    ws = re.sub(r"\s+", " ", t).strip().split(" ")
    if not ws or ws == [""]: return []
    out, step = [], max(1, w - o)
    for s in range(0, len(ws), step):
        out.append(" ".join(ws[s:s + w]))
        if s + w >= len(ws): break
    return out

if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_PATH):
    index = faiss.read_index(INDEX_PATH); CHUNKS = json.load(open(CHUNKS_PATH))
    print('loaded saved index')
else:
    pdfs = sorted(glob.glob(os.path.join(KB_FOLDER, '*.pdf')))
    assert pdfs, f'No PDFs found in {KB_FOLDER}'
    CHUNKS = []
    for f in pdfs:
        doc = fitz.open(f); n0 = len(CHUNKS)
        CHUNKS += chunk("\n".join(doc[pg].get_text() or "" for pg in range(len(doc)))); doc.close()
        print(f'{os.path.basename(f)[:60]:60s} -> {len(CHUNKS) - n0} chunks')
    CHUNKS = [c for c in CHUNKS if c.strip()]
    print(f'{len(CHUNKS)} chunks (the earlier runs had 1,398)')
    vecs = embedder.encode(CHUNKS, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype('float32')
    index = faiss.IndexFlatL2(vecs.shape[1]); index.add(vecs)
    faiss.write_index(index, INDEX_PATH); json.dump(CHUNKS, open(CHUNKS_PATH, 'w'))
    print('index saved to Drive')
assert index.ntotal == len(CHUNKS) and index.d == embedder.get_sentence_embedding_dimension()
print(f'{len(CHUNKS)} chunks, dimension {index.d}')

_rlock = threading.Lock()
def retrieve(query, k=TOP_K, exclude=()):
    """Top-k chunks by L2 distance, skipping chunks already shown in this question."""
    with _rlock:
        q = embedder.encode([query], convert_to_numpy=True).astype('float32')
        _, idx = index.search(q, k + len(exclude))
    out = [CHUNKS[i] for i in idx[0] if i >= 0 and CHUNKS[i] not in exclude]
    return out[:k]

for i, p in enumerate(retrieve('mechanism of foam cell formation in atherosclerotic plaque'), 1):
    print(i, p[:160].replace('\n', ' '), '...')

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

loaded saved index
1398 chunks, dimension 768


/tmp/ipykernel_838/3202266907.py:36: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert index.ntotal == len(CHUNKS) and index.d == embedder.get_sentence_embedding_dimension()


1 the thickness and size of the plaque. Figure 1.8 Atherosclerosis, microscopic At high magnification the necrotic center of an aortic atheroma shows foam cells ( ...
2 risk factors through adoption of a healthy lifestyle with increased exercise and reduced caloric intake can halt the progression of atherosclerosis, and atherom ...
3 R 1 Blood Vessels 3 Figure 1.4 Atherosclerosis, gross This is about as normal as an adult aorta can be. The intimal surface is quite smooth, with only occasiona ...
4 the harbinger of worse atherosclerosis to come, if plaques continue to enlarge. Atherosclerosis is initiated with endothelial damage and inflammation with leuko ...
5 are prone to rupture, hemorrhage, and thrombosis. Platelets become activated and adhere to sites of endothelial injury, then release cytokines such as platelet- ...


## 4. Core code (no changes needed)

In [7]:
import re
def parse_old(response_text):
    """The repository's evaluation/parse_results.py, verbatim (used only for the comparison)."""
    match = re.search(r"(?:^|\n)\s*([A-E])\b", response_text or '')
    if match:
        return match.group(1)
    match = re.search(r"\b([A-E])\b", response_text or '')
    return match.group(1) if match else "PARSE_ERROR"

def parse_new(text, letters):
    """Strict reader: FINAL ANSWER line > 'answer is X' > a letter opening the reply. Never guesses."""
    if not text:
        return None
    t = text.replace('*', '').replace('`', '').replace('#', '')
    m = list(re.finditer(r'FINAL\s+ANSWER\s*[:\-–]?\s*[\[\(]?\s*([A-E])\b', t, re.I))
    if m:
        L = m[-1].group(1).upper()
        return L if L in letters else None
    m = list(re.finditer(r'\b(?:correct\s+)?answer\s*(?:is|:)\s*(?:option\s*)?[\[\(]?\s*([A-E])\b(?![a-z])', t, re.I))
    if m:
        L = m[-1].group(1).upper()
        return L if L in letters else None
    m = re.match(r'\s*[\[\(]?([A-E])[\]\)]?(?:[\.\):\]]|\s*[-–—:]|\s*\n|\s*$|\s+(?=[A-Z]))', t)
    if m and m.group(1) in letters:
        return m.group(1)
    return None


import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm.auto import tqdm
_wlock = threading.Lock()
KEEP = ('tin', 'tout', 'reasoning', 'cached', 'cost', 'sec', 'served')

def make_cfg(label, model, price, key='OPENAI_API_KEY', base_url=None, temperature=0, max_tokens=None):
    return dict(label=label, model=model, price=price, temperature=temperature, max_tokens=max_tokens,
                client=OpenAI(api_key=os.environ[key], base_url=base_url, timeout=600))

RCFG = {name: make_cfg(name, r['model'], r['price'], max_tokens=600) for name, r in READERS.items()}
ASSESSOR = make_cfg(ASSESSOR_MODEL, ASSESSOR_MODEL, ASSESSOR_PRICE, max_tokens=120)

# ---------------------------------------------------------------- money safety
# 1) Every API reply is saved to Drive (api_cache.jsonl) the moment it arrives. If the run stops for any
#    reason, re-running never pays again for a call that was already made: it is read from the cache.
# 2) SPENT = everything paid so far through this notebook (rebuilt from the cache on every start).
#    No new call is made once SPENT reaches MAX_USD.
# 3) If OpenAI reports that the credit is exhausted, the whole run stops at once (no retry loop).
import hashlib
CACHE_PATH = f'{WORKDIR}/api_cache.jsonl'
_clock = threading.Lock()
CACHE, SPENT = {}, 0.0
if os.path.exists(CACHE_PATH):
    for l in open(CACHE_PATH):
        try:
            e = json.loads(l); CACHE[e['key']] = e['res']; SPENT += e['res']['cost']
        except Exception:
            pass          # a half-written last line after a crash is skipped
print(f'cache: {len(CACHE)} saved API replies, US${SPENT:.2f} spent so far; budget US${MAX_USD:.2f}')

class StopRun(Exception):
    pass
STOP = threading.Event()

def _save_line(path, obj):
    with open(path, 'a') as f:
        f.write(json.dumps(obj) + '\n'); f.flush(); os.fsync(f.fileno())

def chat(cfg, messages):
    global SPENT
    kw = dict(model=cfg['model'], messages=messages)
    if cfg.get('temperature') is not None: kw['temperature'] = cfg['temperature']
    if cfg.get('max_tokens'): kw['max_tokens'] = cfg['max_tokens']
    key = hashlib.sha256(json.dumps(kw, sort_keys=True).encode()).hexdigest()
    if key in CACHE:
        return dict(CACHE[key])
    for attempt in range(8):
        if STOP.is_set():
            raise StopRun('run stopped')
        with _clock:
            if SPENT >= MAX_USD:
                STOP.set(); raise StopRun(f'budget of US${MAX_USD:.2f} reached')
        try:
            t0 = time.time()
            r = cfg['client'].chat.completions.create(**kw)
            u = r.usage
            pin, pout = (u.prompt_tokens or 0), (u.completion_tokens or 0)
            billed_out = max(pout, (getattr(u, 'total_tokens', 0) or 0) - pin)    # includes hidden reasoning tokens
            det = getattr(u, 'completion_tokens_details', None)
            reas = (getattr(det, 'reasoning_tokens', 0) or 0) if det is not None else 0
            pdet = getattr(u, 'prompt_tokens_details', None)
            cached = (getattr(pdet, 'cached_tokens', 0) or 0) if pdet is not None else 0   # OpenAI caches repeated prefixes automatically
            p_in, p_out, p_cached = cfg['price']
            res = dict(text=r.choices[0].message.content or '', tin=pin, tout=billed_out, reasoning=reas, cached=cached,
                       cost=((pin - cached) * p_in + cached * p_cached + billed_out * p_out) / 1e6,
                       sec=time.time() - t0, served=getattr(r, 'model', '') or '')
            with _clock:
                CACHE[key] = res; SPENT += res['cost']
                _save_line(CACHE_PATH, dict(key=key, model=cfg['model'], res=res))
            return dict(res)
        except StopRun:
            raise
        except Exception as e:
            msg = str(e); status = getattr(e, 'status_code', None)
            if 'insufficient_quota' in msg or 'billing' in msg.lower() or status in (401, 402, 403):
                STOP.set()
                raise StopRun('OpenAI says the credit is exhausted or the key is not allowed: ' + msg[:200])
            if status is not None and status < 500 and status not in (408, 409, 429):
                raise            # bad request or wrong model name: retrying will not help
            wait = min(2 ** attempt, 60)
            print(f'  [{cfg["label"]}] retry in {wait}s: {type(e).__name__}: {msg[:120]}'); time.sleep(wait)
    raise RuntimeError('API call failed 8 times')

def gap_text(text):
    m = re.search(r'GAP_IDENTIFIED\s*:?\s*\[?(.+)', text or '', re.I | re.S)
    return (m.group(1).strip().split('\n')[0].strip(' []') or 'more information needed')[:300] if m else None

def answer_rec(c, letters):
    return dict(ans=parse_new(c['text'], letters), old=parse_old(c['text']), raw=c['text'][:3000], **{k: c[k] for k in KEEP})

def run_question(q, cfg, with_dec=True):
    sysm = {'role': 'system', 'content': SYSTEM_PROMPT}
    rec = dict(id=q['id'], domain=q['domain'], topic=q.get('topic', ''), key=q['answer'], reader=cfg['label'])
    t, letters = p_norag(q)
    rec['norag'] = answer_rec(chat(cfg, [sysm, {'role': 'user', 'content': t}]), letters)
    p1 = retrieve(q['question'])
    t, _ = p_single(q, p1)
    rec['sp'] = answer_rec(chat(cfg, [sysm, {'role': 'user', 'content': t}]), letters)

    # RAG-SEQ as in the repository prompts: iteration 1 = single-pass context + GAP analysis; later
    # iterations add the gap and new context to the same conversation. A forced answer is requested
    # at every round where a gap is flagged, which gives caps 1, 2 and 3 from one trajectory.
    msgs, shown, rounds, gap = [sysm, {'role': 'user', 'content': p_iter1(q, p1)[0]}], list(p1), [], None
    for r in range(1, CAP + 1):
        if r > 1:
            new = retrieve(q['question'] + ' ' + gap, exclude=set(shown)); shown += new
            msgs.append({'role': 'user', 'content': p_iterN(q, gap, new)[0]})
        c = chat(cfg, msgs)
        gap = gap_text(c['text'])
        step = dict(round=r, gap=gap, **answer_rec(c, letters))
        if gap: step['ans'] = None
        if step['ans'] is None:
            fc = chat(cfg, msgs + [{'role': 'assistant', 'content': c['text']}, {'role': 'user', 'content': FORCE_MSG}])
            step['forced'] = answer_rec(fc, letters)
        rounds.append(step)
        if step['ans'] is not None: break
        msgs.append({'role': 'assistant', 'content': c['text']})
        gap = gap or 'more information needed'
    rec['ga'] = rounds

    if with_dec:   # decoupled: the assessor sees all context so far and decides; the reader answers once (SP prompt)
        steps, ctx = [], list(p1)
        for r in range(1, CAP + 1):
            if r > 1:
                ctx += retrieve(q['question'] + ' ' + steps[-1]['gap'], exclude=set(ctx))
            f = dict(context='\n\n'.join(ctx), question=q['question'], options='\n'.join(f'{L}. {q[L]}' for L in letters))
            c = chat(ASSESSOR, [{'role': 'user', 'content': ASSESSOR_PROMPT.format(**f)}])
            g = gap_text(c['text'])
            steps.append(dict(round=r, gap=g, raw=c['text'][:300], **{k: c[k] for k in KEEP}))
            if not g: break
        t, _ = p_single(q, ctx)
        rec['dec'] = dict(steps=steps, stopped_by_cap=bool(steps[-1]['gap']), n_passages=len(ctx),
                          **answer_rec(chat(cfg, [sysm, {'role': 'user', 'content': t}]), letters))
    return rec

def run_sp_only(q, cfg):
    """Single-pass RAG only (used to fill the frontier models' single-pass column on the primary set)."""
    sysm = {'role': 'system', 'content': SYSTEM_PROMPT}
    t, letters = p_single(q, retrieve(q['question']))
    return dict(id=q['id'], domain=q['domain'], key=q['answer'], reader=cfg['label'],
                sp=answer_rec(chat(cfg, [sysm, {'role': 'user', 'content': t}]), letters))

def run_file(df, out_path, fn, workers=None, limit=None):
    """Runs fn on every question not yet in out_path. Each finished question is written to Drive at once.
    Stops cleanly (keeping everything) when the budget is reached or the credit runs out."""
    if STOP.is_set():
        print('Run is stopped (budget or credit). Add credit / raise MAX_USD, then run the "Resume" cell.'); return
    done = set()
    if os.path.exists(out_path):
        for l in open(out_path):
            try: done.add(json.loads(l)['id'])
            except Exception: pass
    todo = [r for r in df.to_dict('records') if r['id'] not in done][: limit or None]
    print(f'{len(done)} already done, {len(todo)} to run -> {os.path.basename(out_path)}')
    failed = []
    ex = ThreadPoolExecutor(workers or WORKERS)
    futs = {ex.submit(fn, q): q['id'] for q in todo}
    try:
        for fu in tqdm(as_completed(futs), total=len(futs)):
            try:
                rec = fu.result()
            except StopRun as e:
                if not STOP.is_set(): STOP.set()
                failed.append(futs[fu]); continue
            except Exception as e:
                failed.append(futs[fu]); print('FAILED', futs[fu], repr(e)[:200]); continue
            with _wlock:
                _save_line(out_path, rec)
            if STOP.is_set():
                break
    finally:
        ex.shutdown(wait=True, cancel_futures=True)
    n_done = sum(1 for _ in open(out_path)) if os.path.exists(out_path) else 0
    print(f'saved: {n_done} questions in {os.path.basename(out_path)} | spent so far US${SPENT:.2f} of US${MAX_USD:.2f}')
    if STOP.is_set():
        print('STOPPED: budget reached or credit exhausted. Everything above is saved on Drive. '
              'Add credit (or raise MAX_USD), then run the "Resume" cell: it continues where it stopped and does not pay twice.')
    elif failed:
        print(len(failed), 'questions failed - re-run the cell to retry them')

def show(r):
    print(f"\n=== {r['id']} ({r['domain']}) key={r['key']} reader={r['reader']}")
    for c in ('norag', 'sp'):
        if c in r: print(f"  {c.upper():5s} new={r[c]['ans']} old={r[c]['old']} served={r[c]['served']} | {r[c]['raw'][:140]!r}")
    for s in r.get('ga', []):
        print(f"  RAG-SEQ round {s['round']}: new={s['ans']} gap={s['gap']!r} forced={s.get('forced', {}).get('ans')} | {s['raw'][:140]!r}")
    for s in r.get('dec', {}).get('steps', []):
        print(f"  ASSESSOR round {s['round']}: gap={s['gap']!r} | {s['raw'][:80]!r}")
    if 'dec' in r: print(f"  DEC new={r['dec']['ans']} ({r['dec']['n_passages']} passages)")

cache: 11889 saved API replies, US$7.73 spent so far; budget US$9.00


## 5. Runs A and B (the only paid cell)

In [8]:
# =====================================================================================
# Runs A and B. Needs the cells above (Sections 1, 2, 3, 5); all of them are free.
# Nothing already saved is changed: new results and new API replies go to new files in MMIRAG_R3.
# =====================================================================================
import json, os, hashlib, concurrent.futures as cf
import pandas as pd

# New API replies are saved in a separate file, so the original api_cache.jsonl (fingerprinted) is not touched.
spent_before_AB = SPENT if 'spent_before_AB' not in globals() else spent_before_AB   # spending before runs A and B
CACHE_PATH = f'{WORKDIR}/api_cache_runs_A_B.jsonl'
if os.path.exists(CACHE_PATH):                      # after a restart: reuse replies already paid for
    for l in open(CACHE_PATH):
        try:
            e = json.loads(l)
        except Exception:
            continue
        if e['key'] not in CACHE:
            CACHE[e['key']] = e['res']; SPENT += e['res']['cost']

LOG = f'{WORKDIR}/r3_runs_A_B_summary.txt'
def say(*a):
    t = ' '.join(str(x) for x in a); print(t)
    with open(LOG, 'a') as f: f.write(t + '\n')

EXTRA_USD = 0.95                       # the most this notebook may spend on runs A and B
MAX_USD = spent_before_AB + EXTRA_USD
STOP.clear()
print(f'Spent before runs A and B: US${spent_before_AB:.2f}; spent on them so far: US${SPENT - spent_before_AB:.2f}; limit US${EXTRA_USD:.2f}.')

READER = RCFG[MAIN_READER]
JUDGE_B = make_cfg('judge-4o-mini', 'gpt-4o-mini', (0.15, 0.60, 0.075), max_tokens=80)
SYSM = {'role': 'system', 'content': SYSTEM_PROMPT}
FORCE_FINAL = ('Using the question, the options and any context above, choose the single best option now.\n'
               'Format: [LETTER] [Justification], then a last line FINAL ANSWER: <letter>')

def cache_key(cfg, messages):          # identical to the key used by chat(): tells whether a call was made before
    kw = dict(model=cfg['model'], messages=messages)
    if cfg.get('temperature') is not None: kw['temperature'] = cfg['temperature']
    if cfg.get('max_tokens'): kw['max_tokens'] = cfg['max_tokens']
    return hashlib.sha256(json.dumps(kw, sort_keys=True).encode()).hexdigest()

def load_questions(name):
    df = pd.read_csv(f'{WORKDIR}/{name}', keep_default_na=False, dtype=str)
    return {r['id']: r for r in df.to_dict('records')}

def letters_of(q):
    return [L for L in 'ABCDE' if str(q.get(L, '')).strip()]

# ------------------------------------------------------------------ Run A
# For the replies that could not be parsed (almost all bare GAP statements), ask once for a final answer,
# exactly as RAG-SEQ does at the cap. The original messages are rebuilt and checked against the saved replies.
def original_messages(q, rec, cond):
    if cond == 'norag':
        return [SYSM, {'role': 'user', 'content': p_norag(q)[0]}]
    p1 = retrieve(q['question'])
    if cond == 'sp':
        return [SYSM, {'role': 'user', 'content': p_single(q, p1)[0]}]
    ctx = list(p1)                                     # decoupled: rebuild the context from the saved gap statements
    for s in rec['dec']['steps'][:-1]:
        ctx += retrieve(q['question'] + ' ' + s['gap'], exclude=set(ctx))
    return [SYSM, {'role': 'user', 'content': p_single(q, ctx)[0]}]

def run_A(ds_file, q_file):
    out = f'{WORKDIR}/r3A_forced_answers_{ds_file.split("_")[-1]}'
    done = set()
    if os.path.exists(out):
        done = {(json.loads(l)['id'], json.loads(l)['condition']) for l in open(out)}
    Q = load_questions(q_file)
    todo = []
    for l in open(f'{WORKDIR}/{ds_file}'):
        r = json.loads(l)
        for cond in ('norag', 'sp', 'dec'):
            if r[cond]['ans'] is None and (r['id'], cond) not in done:
                todo.append((r, cond))
    print(f'Run A on {ds_file}: {len(todo)} unparsed replies to follow up ({len(done)} already done)')
    for r, cond in todo:
        q = Q[r['id']]
        msgs = original_messages(q, r, cond)
        k0 = cache_key(READER, msgs); same = k0 in CACHE
        prev = CACHE[k0]['text'] if same else r[cond]['raw']          # the full original reply
        c = chat(READER, msgs + [{'role': 'assistant', 'content': prev}, {'role': 'user', 'content': FORCE_FINAL}])
        rec = dict(id=r['id'], condition=cond, key=r['key'], original_call_matched=same, original=r[cond]['raw'][:300],
                   **answer_rec(c, letters_of(q)))
        _save_line(out, rec)
        say(f"  {r['id'][:8]} {cond:5s} original call matched: {same} | forced answer {rec['ans']} (key {r['key']})")
    return out

# ------------------------------------------------------------------ Run B
# Faithfulness of the RAG-SEQ (cap 3) answer on the new outputs, judged by a different model (gpt-4o-mini)
# with the same judge prompt as the earlier audit. The context is everything RAG-SEQ showed the reader.
JUDGE_SYS = ("You assess FAITHFULNESS: is the model's justification supported by the provided context, and does "
             "it support the chosen answer? Reply 'SUPPORTED' or 'UNSUPPORTED' then a brief reason.")

def rag_seq_final(q, rec):
    """Rebuild the RAG-SEQ conversation up to the cap-3 answer; return (context passages, reply, letter, matched)."""
    p1 = retrieve(q['question'])
    msgs, shown, gap = [SYSM, {'role': 'user', 'content': p_iter1(q, p1)[0]}], list(p1), None
    matched = True
    for s in rec['ga'][:CAP]:
        if s['round'] > 1:
            new = retrieve(q['question'] + ' ' + gap, exclude=set(shown)); shown += new
            msgs.append({'role': 'user', 'content': p_iterN(q, gap, new)[0]})
        matched = matched and cache_key(READER, msgs) in CACHE
        if s['ans'] is not None:
            return shown, s['raw'], s['ans'], matched
        if s['round'] == CAP or s is rec['ga'][-1]:
            f = s.get('forced') or {}
            return shown, f.get('raw', ''), f.get('ans'), matched
        msgs.append({'role': 'assistant', 'content': s['raw']})
        gap = s['gap'] or 'more information needed'

def judge_one(args):
    q, rec = args
    shown, reply, letter, matched = rag_seq_final(q, rec)
    opts = '\n'.join(f'{L}. {q[L]}' for L in letters_of(q))
    c = chat(JUDGE_B, [{'role': 'system', 'content': JUDGE_SYS},
                       {'role': 'user', 'content': f"CONTEXT:\n{chr(10).join(shown)}\n\nQUESTION:\n{q['question']}\nOPTIONS:\n{opts}\n"
                                                    f"CHOSEN: {letter}\nJUSTIFICATION:\n{reply}"}])
    t = c['text'].upper()
    return dict(id=rec['id'], domain=rec['domain'], key=rec['key'], answer=letter, correct=letter == rec['key'],
                context_matched=matched, supported=('SUPPORTED' in t and 'UNSUPPORTED' not in t),   # rule of the earlier audit
                judge=c['text'][:300], cost=c['cost'])

def run_B(ds_file, q_file):
    out = f'{WORKDIR}/r3B_faithfulness_{ds_file.split("_")[-1]}'
    done = {json.loads(l)['id'] for l in open(out)} if os.path.exists(out) else set()
    Q = load_questions(q_file)
    recs = [json.loads(l) for l in open(f'{WORKDIR}/{ds_file}')]
    todo = [(Q[r['id']], r) for r in recs if r['id'] not in done]
    print(f'Run B on {ds_file}: {len(todo)} questions to judge ({len(done)} already done)')
    stopped = None
    with cf.ThreadPoolExecutor(WORKERS) as ex:
        futs = [ex.submit(judge_one, a) for a in todo]
        for k, fu in enumerate(cf.as_completed(futs), 1):
            try:
                _save_line(out, fu.result())
            except StopRun as e:
                stopped = e
                continue          # keep saving the judgements that did arrive
            if k % 100 == 0: print(f'  {k}/{len(todo)} | spent so far US${SPENT:.2f}')
    if stopped:
        raise stopped
    return out

def summary_B(path):
    if not os.path.exists(path):
        return say(f'  {os.path.basename(path)}: not run yet')
    R = [json.loads(l) for l in open(path)]
    n = len(R); s = sum(r['supported'] for r in R); w = [r for r in R if not r['correct']]
    say(f"  {os.path.basename(path)}: {n} questions; context rebuilt exactly for {sum(r['context_matched'] for r in R)}")
    say(f"  justification judged supported: {s}/{n} = {s / n:.1%}")
    say(f"  wrong answers: {len(w)}, of which judged supported: {sum(r['supported'] for r in w)}"
          f" = {sum(r['supported'] for r in w) / max(len(w), 1):.1%}")

def summary_A(path, ds_file):
    from scipy.stats import binomtest
    F = {}
    if os.path.exists(path):
        for l in open(path):
            x = json.loads(l); F[(x['id'], x['condition'])] = x['ans']
    R = [json.loads(l) for l in open(f'{WORKDIR}/{ds_file}')]
    def cap3(ga):
        for s in ga[:CAP]:
            if s['ans'] is not None: return s['ans']
            if s['round'] == CAP or s is ga[-1]: return (s.get('forced') or {}).get('ans')
    ok = {'norag': [], 'sp': [], 'dec': [], 'ragseq': []}
    for r in R:
        for c in ('norag', 'sp', 'dec'):
            a = r[c]['ans'] if r[c]['ans'] is not None else F.get((r['id'], c))
            ok[c].append(a == r['key'])
        ok['ragseq'].append(cap3(r['ga']) == r['key'])
    say(f'  {ds_file}: accuracy with the follow-up answers: ' + ', '.join(f'{c} {sum(v) / len(v):.3f}' for c, v in ok.items()))
    for a, b in (('ragseq', 'sp'), ('ragseq', 'norag'), ('sp', 'norag'), ('ragseq', 'dec')):
        g = sum(x and not y for x, y in zip(ok[a], ok[b])); l = sum(y and not x for x, y in zip(ok[a], ok[b]))
        say(f'    {a} vs {b}: {g} gained, {l} lost, exact McNemar p = {binomtest(min(g, l), g + l, 0.5).pvalue if g + l else 1:.4f}')

# ------------------------------------------------------------------ run both
try:
    a_file = run_A(f'r3_{MAIN_READER}_medmcqa.jsonl', 'medmcqa_set.csv')
    summary_A(a_file, f'r3_{MAIN_READER}_medmcqa.jsonl')
    run_A(f'r3_{MAIN_READER}_primary.jsonl', 'clean250.csv')          # the primary set has no unparsed replies
    b_primary = run_B(f'r3_{MAIN_READER}_primary.jsonl', 'clean250.csv')
    summary_B(b_primary)
    b_med = run_B(f'r3_{MAIN_READER}_medmcqa.jsonl', 'medmcqa_set.csv')
    summary_B(b_med)
except StopRun as e:
    say('STOPPED:', e, '- everything finished so far is saved on Drive. Add credit (or raise EXTRA_USD if the limit was '
        'reached) and run all again: it continues where it stopped and pays nothing twice.')
say(f'\nSpent on runs A and B: US${SPENT - spent_before_AB:.2f} (US${SPENT:.2f} in total). Saved in MMIRAG_R3 on Drive: r3A_forced_answers_medmcqa.jsonl, r3B_faithfulness_primary.jsonl, r3B_faithfulness_medmcqa.jsonl, api_cache_runs_A_B.jsonl, r3_runs_A_B_summary.txt')

Spent before runs A and B: US$7.73; spent on them so far: US$0.00; limit US$0.95.
Run A on r3_gpt-4.1-mini_medmcqa.jsonl: 19 unparsed replies to follow up (0 already done)
  98248c80 sp    original call matched: True | forced answer C (key D)
  98248c80 dec   original call matched: True | forced answer C (key D)
  606aa4ad norag original call matched: True | forced answer None (key A)
  606aa4ad sp    original call matched: True | forced answer None (key A)
  606aa4ad dec   original call matched: True | forced answer None (key A)
  996f1f8f norag original call matched: True | forced answer D (key B)
  d544bca5 norag original call matched: True | forced answer A (key A)
  d544bca5 sp    original call matched: True | forced answer None (key A)
  55c6dd5b sp    original call matched: True | forced answer A (key A)
  55c6dd5b dec   original call matched: True | forced answer A (key A)
  a09ccfd1 norag original call matched: True | forced answer A (key A)
  b27b122f sp    original call matc

## 6. Save to Drive and download
Packs the new files into one zip, copies it to `MMIRAG_R3` on Drive, prints each file's SHA-256 fingerprint (for `MANIFEST.md`), downloads the zip and makes sure Drive has received everything.

In [9]:
import zipfile, shutil, hashlib
from google.colab import files, drive
new = [p for p in (f'{WORKDIR}/r3A_forced_answers_medmcqa.jsonl', f'{WORKDIR}/r3B_faithfulness_primary.jsonl',
                   f'{WORKDIR}/r3B_faithfulness_medmcqa.jsonl', f'{WORKDIR}/api_cache_runs_A_B.jsonl',
                   f'{WORKDIR}/r3_runs_A_B_summary.txt') if os.path.exists(p)]
zp = '/content/MMIRAG_R3_runs_A_B.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in new:
        z.write(p, os.path.basename(p))
shutil.copy(zp, f'{WORKDIR}/MMIRAG_R3_runs_A_B.zip')
print('Saved on Drive in MMIRAG_R3:')
for p in new:
    n = sum(1 for _ in open(p))
    print(f'  {os.path.basename(p):36s} {n:6d} lines  sha256 {hashlib.sha256(open(p, "rb").read()).hexdigest()}')
print('  MMIRAG_R3_runs_A_B.zip (all of the above)')
print('original api_cache.jsonl sha256', hashlib.sha256(open(f'{WORKDIR}/api_cache.jsonl', 'rb').read()).hexdigest())
files.download(zp)
drive.flush_and_unmount()     # writes everything still pending to Drive; run all again to remount
print('Drive is up to date.')

Saved on Drive in MMIRAG_R3:
  r3A_forced_answers_medmcqa.jsonl         19 lines  sha256 5716d6430d6eb72378906f721affdf34a400c338dfcd52e95de466de3807bfd9
  r3B_faithfulness_primary.jsonl          250 lines  sha256 629127281a0014c35ca69cda1b757925c5963ea9e16ac00428825c3d4636140c
  r3B_faithfulness_medmcqa.jsonl         1000 lines  sha256 7a8d88b49e39c3ad7e48f4b64bf8cb68ab53d62592bb6ed7f6efff779be5d6e0
  api_cache_runs_A_B.jsonl               1269 lines  sha256 92688f2605a1ca5ba253c1afb1885a0f2572afcb3bddc255e18d8f278dd8ef5b
  r3_runs_A_B_summary.txt                  32 lines  sha256 7ebe9b4d54e7a3300e28672447d63b34d7523ddc9836adcf386f6f72883c85f4
  MMIRAG_R3_runs_A_B.zip (all of the above)
original api_cache.jsonl sha256 63d322ce68b7659548faeb93df66c5df7c3250eff87545e7ddcdf5a56d76b1b0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Drive is up to date.
